In [ ]:
import pandas as pd
import numpy as np


# 1. 读取 installments_payments 数据
installments = pd.read_csv(
    "data/raw/home_credit/installments_payments.csv",
)


# 2. 查看数据规模
print(
    "installments shape:",
    installments.shape
)


# 3. 查看前几行
display(
    installments.head()
)


# 4. 查看字段类型
installments.info()

In [ ]:
print(
    "总还款记录数量:",
    installments.shape[0]
)

print(
    "唯一客户数量:",
    installments["SK_ID_CURR"].nunique()
)

print(
    "唯一历史贷款数量:",
    installments["SK_ID_PREV"].nunique()
)

In [ ]:
installments.duplicated().sum()

In [ ]:
installments.duplicated(
    subset=[
        "SK_ID_PREV",
        "NUM_INSTALMENT_NUMBER"
    ]
).sum()

In [ ]:
installments_missing = (
    installments
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

installments_missing[
    "missing_rate"
] = (
    installments_missing[
        "missing_count"
    ]
    / len(installments)
)

installments_missing

In [ ]:
# ============================================================
# 【特征工程 1】
# 还款时间行为辅助特征
# ============================================================


# 1. 计算实际还款日期与计划还款日期的差值
installments[
    "PAYMENT_DAY_DIFF"
] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
)


# 2. 标记是否迟还
installments[
    "IS_LATE_PAYMENT"
] = (
    installments["PAYMENT_DAY_DIFF"] > 0
).astype("int8")


# 3. 建立迟还天数
installments[
    "LATE_PAYMENT_DAYS"
] = (
    installments["PAYMENT_DAY_DIFF"]
    .clip(lower=0)
)


# 4. 标记是否提前还款
installments[
    "IS_EARLY_PAYMENT"
] = (
    installments["PAYMENT_DAY_DIFF"] < 0
).astype("int8")


# 5. 建立提前还款天数
installments[
    "EARLY_PAYMENT_DAYS"
] = (
    -installments["PAYMENT_DAY_DIFF"]
).clip(lower=0)


# 6. 查看结果
installments[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "NUM_INSTALMENT_NUMBER",
        "DAYS_INSTALMENT",
        "DAYS_ENTRY_PAYMENT",
        "PAYMENT_DAY_DIFF",
        "IS_LATE_PAYMENT",
        "LATE_PAYMENT_DAYS",
        "IS_EARLY_PAYMENT",
        "EARLY_PAYMENT_DAYS"
    ]
].head(10)

In [ ]:
installments[
    [
        "PAYMENT_DAY_DIFF",
        "LATE_PAYMENT_DAYS",
        "EARLY_PAYMENT_DAYS"
    ]
].describe().T

In [ ]:
print(
    "迟还记录数量:",
    installments[
        "IS_LATE_PAYMENT"
    ].sum()
)

print(
    "提前还款记录数量:",
    installments[
        "IS_EARLY_PAYMENT"
    ].sum()
)

In [ ]:
# ============================================================
# 【特征工程 2】
# 还款金额行为特征
# ============================================================


# 1. 计算实际还款金额与应还金额的差值
installments[
    "PAYMENT_AMOUNT_DIFF"
] = (
    installments["AMT_PAYMENT"]
    - installments["AMT_INSTALMENT"]
)


# 2. 标记是否少还
installments[
    "IS_UNDER_PAYMENT"
] = (
    installments["PAYMENT_AMOUNT_DIFF"] < 0
).astype("int8")


# 3. 建立少还金额
installments[
    "UNDER_PAYMENT_AMOUNT"
] = (
    -installments["PAYMENT_AMOUNT_DIFF"]
).clip(lower=0)


# 4. 标记是否多还
installments[
    "IS_OVER_PAYMENT"
] = (
    installments["PAYMENT_AMOUNT_DIFF"] > 0
).astype("int8")


# 5. 建立多还金额
installments[
    "OVER_PAYMENT_AMOUNT"
] = (
    installments["PAYMENT_AMOUNT_DIFF"]
    .clip(lower=0)
)


# 6. 建立实际还款比例
installments[
    "PAYMENT_COMPLETION_RATIO"
] = (
    installments["AMT_PAYMENT"]
    / installments["AMT_INSTALMENT"].replace(0, np.nan)
)


# 7. 查看结果
installments[
    [
        "SK_ID_CURR",
        "SK_ID_PREV",
        "NUM_INSTALMENT_NUMBER",
        "AMT_INSTALMENT",
        "AMT_PAYMENT",
        "PAYMENT_AMOUNT_DIFF",
        "IS_UNDER_PAYMENT",
        "UNDER_PAYMENT_AMOUNT",
        "IS_OVER_PAYMENT",
        "OVER_PAYMENT_AMOUNT",
        "PAYMENT_COMPLETION_RATIO"
    ]
].head(10)

In [ ]:
# 1. 查看数值分布
installments[
    [
        "PAYMENT_AMOUNT_DIFF",
        "UNDER_PAYMENT_AMOUNT",
        "OVER_PAYMENT_AMOUNT",
        "PAYMENT_COMPLETION_RATIO"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 3】
# 分期级还款表现特征
# ============================================================


# 1. 将同一期的多次付款聚合成一条分期记录
installment_level_features = (
    installments
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "NUM_INSTALMENT_VERSION",
            "NUM_INSTALMENT_NUMBER"
        ]
    )
    .agg(
        # 同一期账单的计划还款日期
        INSTALMENT_DUE_DAY=(
            "DAYS_INSTALMENT",
            "max"
        ),

        # 最后一次实际付款日期
        INSTALMENT_LAST_PAYMENT_DAY=(
            "DAYS_ENTRY_PAYMENT",
            "max"
        ),

        # 同一期计划应还金额
        INSTALMENT_DUE_AMOUNT=(
            "AMT_INSTALMENT",
            "max"
        ),

        # 同一期多次实际付款金额之和
        INSTALMENT_PAID_AMOUNT=(
            "AMT_PAYMENT",
            "sum"
        ),

        # 同一期实际付款次数
        INSTALMENT_PAYMENT_COUNT=(
            "AMT_PAYMENT",
            "count"
        )
    )
    .reset_index()
)


# 2. 计算该期最后付款日期与计划还款日期的差值
installment_level_features[
    "INSTALMENT_PAYMENT_DAY_DIFF"
] = (
    installment_level_features[
        "INSTALMENT_LAST_PAYMENT_DAY"
    ]
    - installment_level_features[
        "INSTALMENT_DUE_DAY"
    ]
)


# 3. 标记该期是否迟还
installment_level_features[
    "IS_INSTALMENT_LATE"
] = (
    installment_level_features[
        "INSTALMENT_PAYMENT_DAY_DIFF"
    ] > 0
).astype("int8")


# 4. 建立该期迟还天数
installment_level_features[
    "INSTALMENT_LATE_DAYS"
] = (
    installment_level_features[
        "INSTALMENT_PAYMENT_DAY_DIFF"
    ]
    .clip(lower=0)
)


# 5. 标记该期是否提前还款
installment_level_features[
    "IS_INSTALMENT_EARLY"
] = (
    installment_level_features[
        "INSTALMENT_PAYMENT_DAY_DIFF"
    ] < 0
).astype("int8")


# 6. 建立该期提前还款天数
installment_level_features[
    "INSTALMENT_EARLY_DAYS"
] = (
    -installment_level_features[
        "INSTALMENT_PAYMENT_DAY_DIFF"
    ]
).clip(lower=0)


# 7. 计算该期实际还款金额与应还金额的差值
installment_level_features[
    "INSTALMENT_AMOUNT_DIFF"
] = (
    installment_level_features[
        "INSTALMENT_PAID_AMOUNT"
    ]
    - installment_level_features[
        "INSTALMENT_DUE_AMOUNT"
    ]
)


# 8. 标记该期是否少还
installment_level_features[
    "IS_INSTALMENT_UNDERPAID"
] = (
    installment_level_features[
        "INSTALMENT_AMOUNT_DIFF"
    ] < 0
).astype("int8")


# 9. 建立该期少还金额
installment_level_features[
    "INSTALMENT_UNDERPAID_AMOUNT"
] = (
    -installment_level_features[
        "INSTALMENT_AMOUNT_DIFF"
    ]
).clip(lower=0)


# 10. 标记该期是否多还
installment_level_features[
    "IS_INSTALMENT_OVERPAID"
] = (
    installment_level_features[
        "INSTALMENT_AMOUNT_DIFF"
    ] > 0
).astype("int8")


# 11. 建立该期多还金额
installment_level_features[
    "INSTALMENT_OVERPAID_AMOUNT"
] = (
    installment_level_features[
        "INSTALMENT_AMOUNT_DIFF"
    ]
    .clip(lower=0)
)


# 12. 建立该期还款完成比例
installment_level_features[
    "INSTALMENT_PAYMENT_RATIO"
] = (
    installment_level_features[
        "INSTALMENT_PAID_AMOUNT"
    ]
    / installment_level_features[
        "INSTALMENT_DUE_AMOUNT"
    ].replace(0, np.nan)
)


# 13. 查看分期级结果
installment_level_features.head()

In [ ]:
# 1. 查看数据规模
print(
    "分期级特征 Shape:",
    installment_level_features.shape
)


# 2. 检查分期组合是否仍然重复
print(
    "重复分期数量:",
    installment_level_features.duplicated(
        subset=[
            "SK_ID_PREV",
            "NUM_INSTALMENT_VERSION",
            "NUM_INSTALMENT_NUMBER"
        ]
    ).sum()
)


# 3. 检查非负特征
installment_level_features[
    [
        "INSTALMENT_PAYMENT_COUNT",
        "INSTALMENT_LATE_DAYS",
        "INSTALMENT_EARLY_DAYS",
        "INSTALMENT_UNDERPAID_AMOUNT",
        "INSTALMENT_OVERPAID_AMOUNT"
    ]
].describe().T

In [ ]:
# ============================================================
# 【特征工程 4】
# 客户级还款时间特征
# ============================================================


# 1. 按客户聚合分期记录
installment_time_features = (
    installment_level_features
    .groupby("SK_ID_CURR")
    .agg(
        # 客户历史分期总数
        INSTALMENT_COUNT=(
            "NUM_INSTALMENT_NUMBER",
            "count"
        ),

        # 迟还分期数量
        LATE_INSTALMENT_COUNT=(
            "IS_INSTALMENT_LATE",
            "sum"
        ),

        # 提前还款分期数量
        EARLY_INSTALMENT_COUNT=(
            "IS_INSTALMENT_EARLY",
            "sum"
        ),

        # 平均迟还天数
        AVG_LATE_PAYMENT_DAYS=(
            "INSTALMENT_LATE_DAYS",
            "mean"
        ),

        # 最大迟还天数
        MAX_LATE_PAYMENT_DAYS=(
            "INSTALMENT_LATE_DAYS",
            "max"
        ),

        # 平均提前还款天数
        AVG_EARLY_PAYMENT_DAYS=(
            "INSTALMENT_EARLY_DAYS",
            "mean"
        ),

        # 最大提前还款天数
        MAX_EARLY_PAYMENT_DAYS=(
            "INSTALMENT_EARLY_DAYS",
            "max"
        ),

        # 平均每期被拆分成多少次付款
        AVG_PAYMENT_COUNT_PER_INSTALMENT=(
            "INSTALMENT_PAYMENT_COUNT",
            "mean"
        ),

        # 单期最多拆分付款次数
        MAX_PAYMENT_COUNT_PER_INSTALMENT=(
            "INSTALMENT_PAYMENT_COUNT",
            "max"
        )
    )
    .reset_index()
)


# 2. 建立迟还分期比例
installment_time_features[
    "LATE_INSTALMENT_RATE"
] = (
    installment_time_features[
        "LATE_INSTALMENT_COUNT"
    ]
    / installment_time_features[
        "INSTALMENT_COUNT"
    ]
)


# 3. 建立提前还款分期比例
installment_time_features[
    "EARLY_INSTALMENT_RATE"
] = (
    installment_time_features[
        "EARLY_INSTALMENT_COUNT"
    ]
    / installment_time_features[
        "INSTALMENT_COUNT"
    ]
)


# 4. 查看客户级时间特征
installment_time_features.head()

In [ ]:
# 1. 检查数据规模
print(
    "installment_time_features shape:",
    installment_time_features.shape
)


# 2. 检查唯一客户数量
print(
    "唯一客户数量:",
    installment_time_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户
print(
    "重复客户数量:",
    installment_time_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
installment_time_features[
    [
        "LATE_INSTALMENT_RATE",
        "EARLY_INSTALMENT_RATE"
    ]
].describe().T

In [ ]:
#下面开始特征工程5:客户级还款金额特征 相关

In [ ]:
# ============================================================
# 【清洗步骤】
# 标记具有有效应还金额和实际还款金额的分期
# ============================================================


# 1. 标记该期是否具有完整的还款金额信息
installment_level_features[
    "HAS_VALID_PAYMENT_AMOUNT"
] = (
    installment_level_features[
        "INSTALMENT_DUE_AMOUNT"
    ].notna()
    &
    installment_level_features[
        "INSTALMENT_PAID_AMOUNT"
    ].notna()
).astype("int8")


# 2. 查看有效金额记录数量
print(
    "有效还款金额记录数量:",
    installment_level_features[
        "HAS_VALID_PAYMENT_AMOUNT"
    ].sum()
)

print(
    "缺失还款金额记录数量:",
    (
        installment_level_features[
            "HAS_VALID_PAYMENT_AMOUNT"
        ] == 0
    ).sum()
)

In [ ]:
# ============================================================
# 【特征工程 5】
# 客户级还款金额特征
# ============================================================


# 1. 按客户聚合还款金额表现
installment_amount_features = (
    installment_level_features
    .groupby("SK_ID_CURR")
    .agg(
        # 具有有效金额信息的分期数量
        VALID_PAYMENT_INSTALMENT_COUNT=(
            "HAS_VALID_PAYMENT_AMOUNT",
            "sum"
        ),

        # 客户累计应还金额
        TOTAL_INSTALMENT_DUE_AMOUNT=(
            "INSTALMENT_DUE_AMOUNT",
            "sum"
        ),

        # 客户累计实际支付金额
        TOTAL_INSTALMENT_PAID_AMOUNT=(
            "INSTALMENT_PAID_AMOUNT",
            "sum"
        ),

        # 平均每期应还金额
        AVG_INSTALMENT_DUE_AMOUNT=(
            "INSTALMENT_DUE_AMOUNT",
            "mean"
        ),

        # 平均每期实际支付金额
        AVG_INSTALMENT_PAID_AMOUNT=(
            "INSTALMENT_PAID_AMOUNT",
            "mean"
        ),

        # 少还分期数量
        UNDERPAID_INSTALMENT_COUNT=(
            "IS_INSTALMENT_UNDERPAID",
            "sum"
        ),

        # 多还分期数量
        OVERPAID_INSTALMENT_COUNT=(
            "IS_INSTALMENT_OVERPAID",
            "sum"
        ),

        # 累计少还金额
        TOTAL_UNDERPAID_AMOUNT=(
            "INSTALMENT_UNDERPAID_AMOUNT",
            "sum"
        ),

        # 最大单期少还金额
        MAX_UNDERPAID_AMOUNT=(
            "INSTALMENT_UNDERPAID_AMOUNT",
            "max"
        ),

        # 累计多还金额
        TOTAL_OVERPAID_AMOUNT=(
            "INSTALMENT_OVERPAID_AMOUNT",
            "sum"
        ),

        # 最大单期多还金额
        MAX_OVERPAID_AMOUNT=(
            "INSTALMENT_OVERPAID_AMOUNT",
            "max"
        ),

        # 平均单期还款完成比例
        AVG_INSTALMENT_PAYMENT_RATIO=(
            "INSTALMENT_PAYMENT_RATIO",
            "mean"
        ),

        # 最低单期还款完成比例
        MIN_INSTALMENT_PAYMENT_RATIO=(
            "INSTALMENT_PAYMENT_RATIO",
            "min"
        ),

        # 最高单期还款完成比例
        MAX_INSTALMENT_PAYMENT_RATIO=(
            "INSTALMENT_PAYMENT_RATIO",
            "max"
        )
    )
    .reset_index()
)


# 2. 建立少还分期比例
installment_amount_features[
    "UNDERPAID_INSTALMENT_RATE"
] = (
    installment_amount_features[
        "UNDERPAID_INSTALMENT_COUNT"
    ]
    / installment_amount_features[
        "VALID_PAYMENT_INSTALMENT_COUNT"
    ].replace(0, np.nan)
)


# 3. 建立多还分期比例
installment_amount_features[
    "OVERPAID_INSTALMENT_RATE"
] = (
    installment_amount_features[
        "OVERPAID_INSTALMENT_COUNT"
    ]
    / installment_amount_features[
        "VALID_PAYMENT_INSTALMENT_COUNT"
    ].replace(0, np.nan)
)


# 4. 建立客户整体还款完成比例
installment_amount_features[
    "TOTAL_PAYMENT_COMPLETION_RATIO"
] = (
    installment_amount_features[
        "TOTAL_INSTALMENT_PAID_AMOUNT"
    ]
    / installment_amount_features[
        "TOTAL_INSTALMENT_DUE_AMOUNT"
    ].replace(0, np.nan)
)


# 5. 建立客户累计还款金额差额
installment_amount_features[
    "TOTAL_PAYMENT_AMOUNT_DIFF"
] = (
    installment_amount_features[
        "TOTAL_INSTALMENT_PAID_AMOUNT"
    ]
    - installment_amount_features[
        "TOTAL_INSTALMENT_DUE_AMOUNT"
    ]
)


# 6. 查看客户级还款金额特征
installment_amount_features.head()

In [ ]:
# 1. 检查数据规模和客户唯一性
print(
    "installment_amount_features shape:",
    installment_amount_features.shape
)

print(
    "唯一客户数量:",
    installment_amount_features[
        "SK_ID_CURR"
    ].nunique()
)

print(
    "重复客户数量:",
    installment_amount_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)


# 2. 检查金额特征
installment_amount_features[
    [
        "TOTAL_INSTALMENT_DUE_AMOUNT",
        "TOTAL_INSTALMENT_PAID_AMOUNT",
        "TOTAL_UNDERPAID_AMOUNT",
        "MAX_UNDERPAID_AMOUNT",
        "TOTAL_OVERPAID_AMOUNT",
        "MAX_OVERPAID_AMOUNT"
    ]
].describe().T


# 3. 检查比例特征
installment_amount_features[
    [
        "UNDERPAID_INSTALMENT_RATE",
        "OVERPAID_INSTALMENT_RATE",
        "AVG_INSTALMENT_PAYMENT_RATIO",
        "MIN_INSTALMENT_PAYMENT_RATIO",
        "MAX_INSTALMENT_PAYMENT_RATIO",
        "TOTAL_PAYMENT_COMPLETION_RATIO"
    ]
].describe().T

In [ ]:
## ============================================================
# 【特征工程 6】
# 客户近期一年还款表现特征
# ============================================================


# 1. 筛选申请日前一年内的分期记录
recent_installments = (
    installment_level_features[
        installment_level_features[
            "INSTALMENT_DUE_DAY"
        ] >= -365
    ]
    .copy()
)


# 2. 查看近期数据规模
print(
    "最近一年分期记录数量:",
    recent_installments.shape[0]
)

print(
    "最近一年涉及客户数量:",
    recent_installments[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 按客户聚合近期还款表现
installment_recent_features = (
    recent_installments
    .groupby("SK_ID_CURR")
    .agg(
        # 最近一年分期总数
        RECENT_INSTALMENT_COUNT=(
            "NUM_INSTALMENT_NUMBER",
            "count"
        ),

        # 最近一年迟还次数
        RECENT_LATE_INSTALMENT_COUNT=(
            "IS_INSTALMENT_LATE",
            "sum"
        ),

        # 最近一年平均迟还天数
        RECENT_AVG_LATE_PAYMENT_DAYS=(
            "INSTALMENT_LATE_DAYS",
            "mean"
        ),

        # 最近一年最大迟还天数
        RECENT_MAX_LATE_PAYMENT_DAYS=(
            "INSTALMENT_LATE_DAYS",
            "max"
        ),

        # 最近一年少还次数
        RECENT_UNDERPAID_INSTALMENT_COUNT=(
            "IS_INSTALMENT_UNDERPAID",
            "sum"
        ),

        # 最近一年有效还款金额记录数量
        RECENT_VALID_PAYMENT_COUNT=(
            "HAS_VALID_PAYMENT_AMOUNT",
            "sum"
        ),

        # 最近一年累计应还金额
        RECENT_TOTAL_DUE_AMOUNT=(
            "INSTALMENT_DUE_AMOUNT",
            "sum"
        ),

        # 最近一年累计实际支付金额
        RECENT_TOTAL_PAID_AMOUNT=(
            "INSTALMENT_PAID_AMOUNT",
            "sum"
        ),

        # 最近一年平均单期还款完成比例
        RECENT_AVG_PAYMENT_RATIO=(
            "INSTALMENT_PAYMENT_RATIO",
            "mean"
        ),

        # 最近一年最低单期还款完成比例
        RECENT_MIN_PAYMENT_RATIO=(
            "INSTALMENT_PAYMENT_RATIO",
            "min"
        )
    )
    .reset_index()
)


# 4. 建立最近一年迟还比例
installment_recent_features[
    "RECENT_LATE_INSTALMENT_RATE"
] = (
    installment_recent_features[
        "RECENT_LATE_INSTALMENT_COUNT"
    ]
    / installment_recent_features[
        "RECENT_INSTALMENT_COUNT"
    ].replace(0, np.nan)
)


# 5. 建立最近一年少还比例
installment_recent_features[
    "RECENT_UNDERPAID_INSTALMENT_RATE"
] = (
    installment_recent_features[
        "RECENT_UNDERPAID_INSTALMENT_COUNT"
    ]
    / installment_recent_features[
        "RECENT_VALID_PAYMENT_COUNT"
    ].replace(0, np.nan)
)


# 6. 建立最近一年整体还款完成比例
installment_recent_features[
    "RECENT_TOTAL_PAYMENT_COMPLETION_RATIO"
] = (
    installment_recent_features[
        "RECENT_TOTAL_PAID_AMOUNT"
    ]
    / installment_recent_features[
        "RECENT_TOTAL_DUE_AMOUNT"
    ].replace(0, np.nan)
)


# 7. 查看客户近期还款特征
installment_recent_features.head()

In [ ]:
# 1. 检查客户唯一性
print(
    "installment_recent_features shape:",
    installment_recent_features.shape
)

print(
    "重复客户数量:",
    installment_recent_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)


# 2. 检查比例范围
installment_recent_features[
    [
        "RECENT_LATE_INSTALMENT_RATE",
        "RECENT_UNDERPAID_INSTALMENT_RATE",
        "RECENT_AVG_PAYMENT_RATIO",
        "RECENT_MIN_PAYMENT_RATIO",
        "RECENT_TOTAL_PAYMENT_COMPLETION_RATIO"
    ]
].describe().T


# 3. 检查天数和金额
installment_recent_features[
    [
        "RECENT_AVG_LATE_PAYMENT_DAYS",
        "RECENT_MAX_LATE_PAYMENT_DAYS",
        "RECENT_TOTAL_DUE_AMOUNT",
        "RECENT_TOTAL_PAID_AMOUNT"
    ]
].describe().T

In [ ]:
# 【检查步骤】
# 检查同一期是否存在多个不同的应还金额

due_amount_check = (
    installments
    .groupby(
        [
            "SK_ID_CURR",
            "SK_ID_PREV",
            "NUM_INSTALMENT_VERSION",
            "NUM_INSTALMENT_NUMBER"
        ]
    )["AMT_INSTALMENT"]
    .nunique()
)

print(
    "同一期存在多个不同应还金额的数量:",
    (due_amount_check > 1).sum()
)

due_amount_check.describe()

In [ ]:
# ============================================================
# 【合并步骤】
# 合并 installments_payments 客户级特征
# ============================================================


# 1. 以客户级还款时间特征作为主表
installments_features = (
    installment_time_features

    # 2. 合并客户级还款金额特征
    .merge(
        installment_amount_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    # 3. 合并客户近期一年还款特征
    .merge(
        installment_recent_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)


# 4. 查看最终结果
installments_features.head()

In [ ]:
# 1. 查看最终数据规模
print(
    "installments_features shape:",
    installments_features.shape
)


# 2. 检查唯一客户数量
print(
    "唯一客户数量:",
    installments_features[
        "SK_ID_CURR"
    ].nunique()
)


# 3. 检查重复客户数量
print(
    "重复客户数量:",
    installments_features[
        "SK_ID_CURR"
    ].duplicated().sum()
)

In [ ]:
duplicate_suffix_columns = [
    col
    for col in installments_features.columns
    if col.endswith("_x") or col.endswith("_y")
]

duplicate_suffix_columns

In [ ]:
installments_missing_summary = (
    installments_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)


installments_missing_summary[
    "missing_rate"
] = (
    installments_missing_summary[
        "missing_count"
    ]
    / len(installments_features)
)


installments_missing_summary.head(20)